### Model 1: ResNet-18 Acoustic Branch

### **Purpose:** Process SPRSound audio data to extract acoustic features and predict respiratory distress symptoms
 
### **Architecture:** ResNet-18 (1D adapted) with Quantization-Aware Training (QAT)

### **Output:** P(y|X_audio) - probability distribution over 6 symptom classes

## 1. Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.append('..')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories
Path("../models").mkdir(exist_ok=True)

Using device: cpu


## 2. Load Enriched Data

In [2]:
X_train = np.load("../../sound_data/model_data/X_train.npy")
X_val = np.load("../../sound_data/model_data/X_val.npy")
X_test = np.load("../../sound_data/model_data/X_test.npy")

y_train_symptom = np.load("../../sound_data/model_data/y_train_symptom.npy", allow_pickle=True)
y_val_symptom = np.load("../../sound_data/model_data/y_val_symptom.npy", allow_pickle=True)
y_test_symptom = np.load("../../sound_data/model_data/y_test_symptom.npy", allow_pickle=True)

print(f"Data Loaded! Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")


Data Loaded! Train shape: (4596, 33), Val shape: (985, 33), Test shape: (986, 33)


## 3. Label Encodng & Weight Calculation

In [3]:
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train_symptom)
y_val_enc = label_encoder.transform(y_val_symptom)
y_test_enc = label_encoder.transform(y_test_symptom)

print("\nTraining Class Distribution:")
for i, name in enumerate(label_encoder.classes_):
    print(f"  {name}: {np.sum(y_train_enc == i)} samples")

# Calculate weights for Focal Loss
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_enc), y=y_train_enc)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)


Training Class Distribution:
  crackles: 759 samples
  other: 3324 samples
  rhonchi: 54 samples
  stridor: 21 samples
  wheeze: 384 samples
  wheeze_crackle: 54 samples


## 4. Create Data Loaders

In [4]:
batch_size = 64

train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train_enc))
val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val_enc))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test_enc))

# Standard shuffling is now safe because we physically augmented the data!
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

## 5. Define 1D ResNet-18 and Focal Loss

In [7]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class ResNet18_1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)
        
    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        layers = []
        layers.append(ResidualBlock1D(in_channels, out_channels, stride, downsample))
        for _ in range(1, blocks):
            layers.append(ResidualBlock1D(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.unsqueeze(1) # Add channel dimension for 1D Conv
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss) 
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Initialize Model, Loss, and Optimizer
input_features = X_train.shape[1]
num_classes = len(label_encoder.classes_)
model = ResNet18_1D(input_dim=input_features, num_classes=num_classes).to(device)

criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

## 6. Training Loop

In [8]:
n_epochs = 40
best_acc = 0
patience_counter = 0

print("\nStarting Training...")
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    
    for data, labels in train_loader:
        data, labels = data.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, labels in val_loader:
            data, labels = data.to(device), labels.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_acc = 100 * correct / total
    scheduler.step(val_acc)
    
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "../models/model_1_resnet18_best.pth")
    else:
        patience_counter += 1
        
    print(f'Epoch {epoch+1:2d} | Loss: {epoch_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]["lr"]:.2e}')
    
    if patience_counter >= 10:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break


Starting Training...
Epoch  1 | Loss: 1.5897 | Val Acc: 5.99% | LR: 5.00e-04
Epoch  2 | Loss: 1.3512 | Val Acc: 1.42% | LR: 5.00e-04
Epoch  3 | Loss: 1.3055 | Val Acc: 7.51% | LR: 5.00e-04
Epoch  4 | Loss: 1.0812 | Val Acc: 9.44% | LR: 5.00e-04
Epoch  5 | Loss: 0.8963 | Val Acc: 5.08% | LR: 5.00e-04
Epoch  6 | Loss: 0.6237 | Val Acc: 11.68% | LR: 5.00e-04
Epoch  7 | Loss: 0.7593 | Val Acc: 8.12% | LR: 5.00e-04
Epoch  8 | Loss: 0.6801 | Val Acc: 8.73% | LR: 5.00e-04
Epoch  9 | Loss: 0.4698 | Val Acc: 6.80% | LR: 5.00e-04
Epoch 10 | Loss: 0.3285 | Val Acc: 10.46% | LR: 5.00e-04
Epoch 11 | Loss: 0.4370 | Val Acc: 10.86% | LR: 2.50e-04
Epoch 12 | Loss: 0.3371 | Val Acc: 10.25% | LR: 2.50e-04
Epoch 13 | Loss: 0.1871 | Val Acc: 10.25% | LR: 2.50e-04
Epoch 14 | Loss: 0.1451 | Val Acc: 11.07% | LR: 2.50e-04
Epoch 15 | Loss: 0.1110 | Val Acc: 13.91% | LR: 2.50e-04
Epoch 16 | Loss: 0.0821 | Val Acc: 12.89% | LR: 2.50e-04
Epoch 17 | Loss: 0.0702 | Val Acc: 14.21% | LR: 2.50e-04
Epoch 18 | Loss: 

## 7. Final Evaluation on Test Set

In [9]:
print("\nLoading best model for Test Evaluation...")
model.load_state_dict(torch.load("../models/model_1_resnet18_best.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for data, labels in test_loader:
        data, labels = data.to(device), labels.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n" + "="*50)
print("FINAL CLASSIFICATION REPORT (Real Test Data Only)")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))


Loading best model for Test Evaluation...

FINAL CLASSIFICATION REPORT (Real Test Data Only)
                precision    recall  f1-score   support

      crackles       0.16      0.44      0.24       143
         other       0.74      0.40      0.52       736
       rhonchi       0.00      0.00      0.00        12
       stridor       0.00      0.00      0.00         2
        wheeze       0.08      0.16      0.11        85
wheeze_crackle       0.00      0.00      0.00         8

      accuracy                           0.38       986
     macro avg       0.16      0.17      0.14       986
  weighted avg       0.59      0.38      0.43       986

